In [21]:
import sys
sys.path.append(r'C:\Users\liuyujie714\Desktop\TprParser\x64\Release')

import subprocess, shutil, copy
import TprParser
import numpy as np

class TprReader:
    """ @brief A wrapper of TprParser
        1. get atmic coordinates of tpr
        2. modify simulation nsteps, delta t or coordinates and save as new.tpr
    """
    def __init__(self, fname, bGRO = False, bMol2 = False, bCharge = False) -> None:
        # get internal object
        self.tprCapsule = TprParser.load(fname, bGRO, bMol2, bCharge)
    
    def set_nsteps(self, nsteps):
        """ @brief set up nsteps of tpr, same as mdp

        Parameters
        ----------
        nsteps: the nsteps of simulation

        Returns
        -------
        return True if succeed
        """
        return TprParser.set_nsteps(self.tprCapsule, nsteps)

    def set_dt(self, dt):
        """ @brief set up dt of tpr in ps, same as mdp

        Parameters
        ----------
        dt: the dt of simulation, ps

        Returns
        -------
        return True if succeed
        """
        return TprParser.set_dt(self.tprCapsule, dt)
    
    def set_coords(self, coords):
        """ @brief set up atomic coordinates of tpr

        Parameters
        ----------
        coords: a list of atom coordinates, the length must be natoms * 3

        Returns
        -------
        return True if succeed
        """
        return TprParser.set_coordinates(self.tprCapsule, coords)

    def get_coords(self):
        """ @brief get atomic coordinates from tpr

        Returns
        -------
        return a list of atom coordinates, the length is natoms * 3
        """
        return TprParser.get_coordinates(self.tprCapsule)


In [23]:
def run_cmd(cmd:list):
    ret = subprocess.run(cmd, shell=True)
    if ret.returncode != 0:
        raise Exception('\nError occurred from command: \n\t%s!!!' %cmd)
    
def MD(inittpr:str, nsteps:int = 10):
    reader = TprReader(inittpr)
    x = reader.get_coords() # get coords from tpr
    natmA = 120
    natmB = 132
    natm = natmA+natmB

    coords = np.array(x).reshape(-1, 3) # to N*3 shape
    assert natm == coords.shape[0]
    for i in range(nsteps):
        # move two molecules distance of Z axis each 2.0 A
        tempcoords = copy.deepcopy(coords)
        tempcoords[:natmA,     2] += 0.05 * i
        tempcoords[natmA:natm, 2] -= 0.05 * i
        reader.set_coords(np.array(tempcoords.flatten(), dtype=np.float32))
        # rename new.tpr to em_{i}.tpr
        suffix = inittpr.split(".tpr")[0]+"_"+str(i)
        shutil.move("new.tpr", f"{suffix}.tpr")
        run_cmd(f'gmx mdrun -deffnm {suffix} -v')
    print("Finished!")

if __name__ == '__main__':
    MD("em.tpr", 20)


Finished!
